# compute_seasonal_thresholds notebook

This notebook embeds the full code from `compute_seasonal_thresholds.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

import argparse
from pathlib import Path

import numpy as np
import pandas as pd


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Compute variable daily thresholds (Q20, Q50, Q80, Q90) per basin using only "
            "the most recent 30 years of modeled discharge."
        )
    )
    parser.add_argument(
        "--q_csv",
        default="Results/discharge_components/Q_total_all_basins.csv",
        help="Wide CSV with columns: date, basin_1, basin_2, ...",
    )
    parser.add_argument(
        "--years",
        type=int,
        default=30,
        help="Number of most recent years to use (default: 30).",
    )
    parser.add_argument(
        "--out_dir",
        default="Results/thresholds",
        help="Output folder for threshold files.",
    )
    parser.add_argument(
        "--smooth_window_days",
        type=int,
        default=30,
        help="Window size (days) for circular moving-average smoothing (default: 30).",
    )
    return parser.parse_known_args()[0]


def compute_daily_thresholds(df: pd.DataFrame, basin_cols: list[str]) -> pd.DataFrame:
    """
    Compute Q20, Q50, Q80, and Q90 seasonal thresholds by day-of-year.

    Convention used (same as Q80 / Q90):
    - Q90: flow exceeded 90%% of the time -> 10th percentile (q=0.10)
    - Q80: flow exceeded 80%% of the time -> 20th percentile (q=0.20)
    - Q50: flow exceeded 50%% of the time -> 50th percentile (q=0.50)
    - Q20: flow exceeded 20%% of the time -> 80th percentile (q=0.80)
    """
    work = df.copy()
    work["month"] = work["date"].dt.month
    work["day"] = work["date"].dt.day

    records: list[dict] = []
    for basin_id in basin_cols:
        sub = work[["month", "day", basin_id]].rename(columns={basin_id: "q"})
        for (month, day), g in sub.groupby(["month", "day"], sort=True):
            q = pd.to_numeric(g["q"], errors="coerce").to_numpy(dtype=float)
            q = q[np.isfinite(q)]
            if q.size == 0:
                q20 = np.nan
                q50 = np.nan
                q80 = np.nan
                q90 = np.nan
            else:
                q20 = float(np.quantile(q, 0.80))
                q50 = float(np.quantile(q, 0.50))
                q80 = float(np.quantile(q, 0.20))
                q90 = float(np.quantile(q, 0.10))

            month = int(month)
            day = int(day)
            doy = int(pd.Timestamp(year=2000, month=month, day=day).dayofyear)
            records.append(
                {
                    "basin_id": basin_id,
                    "doy": int(doy),
                    "month": month,
                    "day": day,
                    "ddmm": f"{day:02d}-{month:02d}",
                    "Q20": q20,
                    "Q50": q50,
                    "Q80": q80,
                    "Q90": q90,
                    "n_samples": int(q.size),
                }
            )

    out = pd.DataFrame.from_records(records)
    out = out.sort_values(["basin_id", "doy"]).reset_index(drop=True)
    return out


def smooth_climatology_wide(df_wide: pd.DataFrame, window_days: int) -> pd.DataFrame:
    """
    Apply circular moving-average smoothing to a ddmm-indexed climatology table.
    """
    if window_days <= 1:
        return df_wide.copy()

    out = df_wide.copy()
    value_cols = [c for c in out.columns if c != "ddmm"]
    n = len(out)
    if n == 0:
        return out

    for col in value_cols:
        s = pd.to_numeric(out[col], errors="coerce")
        extended = pd.concat([s, s, s], ignore_index=True)
        smoothed = extended.rolling(window=window_days, center=True, min_periods=1).mean()
        out[col] = smoothed.iloc[n : 2 * n].to_numpy()

    return out


def main() -> None:
    args = parse_args()

    q_path = Path(args.q_csv)
    if not q_path.exists():
        raise FileNotFoundError(f"Discharge file not found: {q_path}")

    df = pd.read_csv(q_path)
    if "date" not in df.columns:
        raise ValueError(f"Expected 'date' column in {q_path}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    basin_cols = [c for c in df.columns if c != "date"]
    if not basin_cols:
        raise ValueError("No basin discharge columns found.")

    end_date = df["date"].max()
    start_date = end_date - pd.DateOffset(years=int(args.years))
    recent = df[df["date"] >= start_date].copy()
    if recent.empty:
        raise ValueError("No rows remain after last-N-years filtering.")

    thresholds = compute_daily_thresholds(recent, basin_cols)
    mean_work = recent.copy()
    mean_work["month"] = mean_work["date"].dt.month
    mean_work["day"] = mean_work["date"].dt.day
    mean_q_daily = (
        mean_work.groupby(["month", "day"], sort=True)[basin_cols]
        .mean(numeric_only=True)
        .reset_index()
    )
    mean_q_daily["ddmm"] = mean_q_daily.apply(
        lambda r: f"{int(r['day']):02d}-{int(r['month']):02d}", axis=1
    )
    mean_q_daily = mean_q_daily[["ddmm"] + basin_cols]
    mean_q_daily["_sort"] = pd.to_datetime(mean_q_daily["ddmm"], format="%d-%m", errors="coerce")
    mean_q_daily = mean_q_daily.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    all_out = out_dir / f"seasonal_thresholds_q20_q50_q80_q90_last{args.years}y.csv"
    thresholds.to_csv(all_out, index=False)

    def pivot_sort_smooth(value_col: str) -> pd.DataFrame:
        wide = thresholds.pivot(index="ddmm", columns="basin_id", values=value_col).reset_index()
        wide["_sort"] = pd.to_datetime(wide["ddmm"], format="%d-%m", errors="coerce")
        wide = wide.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
        return smooth_climatology_wide(wide, window_days=int(args.smooth_window_days))

    q20_wide = pivot_sort_smooth("Q20")
    q50_wide = pivot_sort_smooth("Q50")
    q80_wide = pivot_sort_smooth("Q80")
    q90_wide = pivot_sort_smooth("Q90")
    mean_q_daily = smooth_climatology_wide(mean_q_daily, window_days=int(args.smooth_window_days))
    q20_wide.to_csv(out_dir / f"Q20_daily_thresholds_last{args.years}y.csv", index=False)
    q50_wide.to_csv(out_dir / f"Q50_daily_thresholds_last{args.years}y.csv", index=False)
    q80_wide.to_csv(out_dir / f"Q80_daily_thresholds_last{args.years}y.csv", index=False)
    q90_wide.to_csv(out_dir / f"Q90_daily_thresholds_last{args.years}y.csv", index=False)
    mean_q_out = out_dir / f"mean_q_daily_all_basins_last{args.years}y.csv"
    mean_q_daily.to_csv(mean_q_out, index=False)

    print(f"Input: {q_path}")
    print(f"Date window used: {recent['date'].min().date()} to {recent['date'].max().date()}")
    print(f"Smoothing: circular {args.smooth_window_days}-day moving average")
    print(f"Saved: {all_out}")
    print(f"Saved: {out_dir / f'Q20_daily_thresholds_last{args.years}y.csv'}")
    print(f"Saved: {out_dir / f'Q50_daily_thresholds_last{args.years}y.csv'}")
    print(f"Saved: {out_dir / f'Q80_daily_thresholds_last{args.years}y.csv'}")
    print(f"Saved: {out_dir / f'Q90_daily_thresholds_last{args.years}y.csv'}")
    print(f"Saved: {mean_q_out}")


# Execute the script entry point
main()
